In [1]:
import os

import imageio
import kagglehub
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from torchvision import datasets
from torchvision import transforms
import torch.nn.functional as F



%matplotlib inline

In [7]:
batch_size = 32
# MNIST Dataset
train_dataset = datasets.MNIST(root='./mnist_data/', train=True, transform=transforms.ToTensor(), download=False)
test_dataset = datasets.MNIST(root='./mnist_data/', train=False, transform=transforms.ToTensor(), download=False)

# Data Loader (Input Pipeline)
train_loader = torch.utils.data.DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
test_loader = torch.utils.data.DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=False)

## 2.3. Conditional VAE (6 баллов)


Мы уже научились обучать обычный AE на датасете картинок и получать новые картинки, используя генерацию шума и декодер.
Давайте теперь допустим, что мы обучили AE на датасете MNIST и теперь хотим генерировать новые картинки с числами с помощью декодера (как выше мы генерили рандомные лица).
И вот нам понадобилось сгенерировать цифру 8, и мы подставляем разные варианты шума, но восьмерка никак не генерится:(

Хотелось бы добавить к нашему AE функцию "выдай мне рандомное число из вот этого вот класса", где классов десять (цифры от 0 до 9 образуют десять классов).  Conditional AE — так называется вид автоэнкодера, который предоставляет такую возможность. Ну, название "conditional" уже говорит само за себя.

И в этой части задания мы научимся такие обучать.

### Архитектура

На картинке ниже представлена архитектура простого Conditional VAE.

По сути, единственное отличие от обычного -- это то, что мы вместе с картинкой в первом слое энкодера и декодера передаем еще информацию о классе картинки.

То есть, в первый (входной) слой энкодера подается конкатенация картинки и информации о классе (например, вектора из девяти нулей и одной единицы). В первый слой декодера подается конкатенация латентного вектора и информации о классе.


![alt text](https://sun9-63.userapi.com/impg/Mh1akf7mfpNoprrSWsPOouazSmTPMazYYF49Tw/djoHNw_9KVA.jpg?size=1175x642&quality=96&sign=e88baec5f9bb91c8443fba31dcf0a4df&type=album)

![alt text](https://sun9-73.userapi.com/impg/UDuloLNKhzTBYAKewgxke5-YPsAKyGOqA-qCRg/MnyCavJidxM.jpg?size=1229x651&quality=96&sign=f2d21bfacc1c5755b76868dc4cfef39c&type=album)



На всякий случай: это VAE, то есть, latent у него все еще состоит из mu и sigma

Таким образом, при генерации новой рандомной картинки мы должны будем передать декодеру сконкатенированные латентный вектор и класс картинки.

P.S. Также можно передавать класс картинки не только в первый слой, но и в каждый слой сети. То есть на каждом слое конкатенировать выход из предыдущего слоя и информацию о классе.

In [19]:
class CVAE(nn.Module):
    def __init__(self, latent_dim=2):
        super(CVAE, self).__init__()
        
        # Encoder architecture for MNIST (28x28 -> 7x7)
        self.encoder = nn.Sequential(
            nn.Conv2d(11, 32, 4, stride=2, padding=1),  # 1 channel + 10 one-hot classes
            nn.ReLU(),
            nn.Conv2d(32, 64, 4, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 128, 4, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(128, 256, 4, stride=2, padding=1),  # Добавляем еще один слой
            nn.ReLU(),
            nn.Flatten()
        )
        
        # Calculate the size of flattened features (256 * 2 * 2 = 1024)
        self.flat_features = 256 * 2 * 2
        
        # Two separate linear layers for mu and logsigma
        self.fc_mu = nn.Linear(self.flat_features, latent_dim)
        self.fc_logsigma = nn.Linear(self.flat_features, latent_dim)
        
        # Decoder architecture
        self.decoder_input = nn.Linear(latent_dim + 10, self.flat_features)
        
        self.decoder = nn.Sequential(
            nn.Unflatten(1, (256, 2, 2)),
            nn.ConvTranspose2d(256, 128, 4, stride=2, padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(128, 64, 4, stride=2, padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(64, 32, 4, stride=2, padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(32, 1, 4, stride=2, padding=1),
            nn.Sigmoid()  # для MNIST значения пикселей в диапазоне [0,1]
        )

    def encode(self, x, class_num):
        # One-hot encode the class number
        batch_size = x.size(0)
        class_one_hot = F.one_hot(class_num, num_classes=10).float()
        
        # Expand class_one_hot to match image dimensions
        class_one_hot = class_one_hot.view(batch_size, 10, 1, 1)
        class_one_hot = class_one_hot.expand(-1, -1, x.size(2), x.size(3))
        
        # Concatenate input image with class information
        x = torch.cat([x, class_one_hot], dim=1)
        
        # Get features from encoder
        features = self.encoder(x)
        
        # Get mu and logsigma
        mu = self.fc_mu(features)
        logsigma = self.fc_logsigma(features)
        
        return mu, logsigma, class_num

    def gaussian_sampler(self, mu, logsigma):
        if self.training:
            # Sample from normal distribution
            std = torch.exp(0.5 * logsigma)
            eps = torch.randn_like(std)
            return mu + eps * std
        else:
            # Return mu during inference
            return mu

    def decode(self, z, class_num):
        # One-hot encode the class number
        class_one_hot = F.one_hot(class_num, num_classes=10).float()
        
        # Concatenate latent vector with class information
        z = torch.cat([z, class_one_hot], dim=1)
        
        # Decode
        x = self.decoder_input(z)
        reconstruction = self.decoder(x.view(-1, 256, 2, 2))
        
        return reconstruction

    def forward(self, x, class_num):
        mu, logsigma, class_num = self.encode(x, class_num)
        z = self.gaussian_sampler(mu, logsigma)
        reconstruction = self.decode(z, class_num)
        return mu, logsigma, reconstruction

Обучение

In [20]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [21]:

def train_cvae(cvae, data, epochs=20):
    opt = torch.optim.Adam(cvae.parameters())
    cvae.train()  # Переводим модель в режим обучения

    for epoch in range(epochs):
        epoch_loss = 0.0
        num_batches = 0

        for x, y in data:
            x = x.to(device)  # Перемещаем данные на GPU
            y = y.to(device)  # Перемещаем метки на GPU
            
            opt.zero_grad()
            
            # Получаем mu, logsigma и реконструкцию
            mu, logsigma, reconstruction = cvae(x, y)
            
            # Реконструкционная ошибка (MSE)
            recon_loss = F.mse_loss(reconstruction, x, reduction='sum')
            
            # KL дивергенция
            kl_loss = -0.5 * torch.sum(1 + logsigma - mu.pow(2) - logsigma.exp())
            
            # Общий loss
            loss = recon_loss + kl_loss
            
            loss.backward()
            opt.step()

            epoch_loss += loss.item()
            num_batches += 1

        # Выводим средний loss за эпоху
        avg_loss = epoch_loss / num_batches
        print(f'Epoch: {epoch}, Loss: {avg_loss:.4f}')

    return cvae

In [22]:
vae = CVAE().to(device) # GPU
vae = train_cvae(vae, train_loader)

RuntimeError: mat1 and mat2 shapes cannot be multiplied (32x256 and 1024x2)

### Sampling


Тут мы будем сэмплировать из CVAE. Это прикольнее, чем сэмплировать из простого AE/VAE: тут можно взять один и тот же латентный вектор и попросить CVAE восстановить из него картинки разных классов!
Для MNIST вы можете попросить CVAE восстановить из одного латентного вектора, например, картинки цифры 5 и 7.

In [ ]:
< тут
нужно
научиться
сэмплировать
из
декодера
цифры
определенного
класса >

Splendid! Вы великолепны!


### Latent Representations

Давайте посмотрим, как выглядит латентное пространство картинок в CVAE и сравним с картинкой для VAE =)

Опять же, нужно покрасить точки в разные цвета в зависимости от класса.

In [ ]:
< ваш
код
получения
латентных
представлений, применения
TSNE
и
визуализации >

Что вы думаете насчет этой картинки? Отличается от картинки для VAE?